# Multi-Agent AI Analyst — Main Notebook

This notebook just wires the modules together and runs the graph. All logic lives in `config.py`, `state.py`, `data_setup.py`, `vectorstore.py`, `agents.py`, and `graph.py`.

**If you edit any node function, re-run the "Build graph" cell** — LangGraph doesn't pick up redefinitions of already-compiled nodes.

**Setup note:** all six module files (`config.py`, `state.py`, `data_setup.py`, `vectorstore.py`, `agents.py`, `graph.py`) must sit in the same folder as this notebook, and `agents.py` must be named exactly that — `graph.py` does `from agents import ...`, so a file named `agentes.py` (or anything else) will fail to import.

## 1. Install dependencies

In [4]:
!pip install -q langchain langgraph langchain-openai langchain-community qdrant-client langchain-text-splitters tavily-python pandas


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1b. Sanity check — required module files present

In [5]:
import pathlib

required = ["config.py", "state.py", "data_setup.py", "vectorstore.py", "agents.py", "graph.py"]
missing = [f for f in required if not pathlib.Path(f).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing required file(s) next to this notebook: {missing}. "
        "Check for typos in the filename (e.g. agents.py, not agentes.py)."
    )
print("All required module files found.")

All required module files found.


## 2. Load API keys and LLMs

In [6]:
from config import get_llms, load_tavily_key

llm_flash, llm_lite = get_llms()
load_tavily_key()  # optional, web_agent skips gracefully if left blank

print(llm_flash.invoke("Say hello in one short sentence.").content)

Gemini key loaded: True
Tavily key loaded: True
Hello, it is nice to meet you.


## 3. Load data (SQLite + churn report)

In [7]:
from data_setup import setup_data, build_churn_report, DB_PATH
from langchain_community.utilities import SQLDatabase

df = setup_data()
db = SQLDatabase.from_uri(f"sqlite:///{DB_PATH}")

C:\Users\user\AppData\Local\Temp\ipykernel_21368\1619636803.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


Shape: (7032, 21)
Churn
No     5163
Yes    1869
Name: count, dtype: int64
Customer Churn Analysis Report (derived from real customer data)

Churn rate by contract type:
Contract
Month-to-month    42.7
One year          11.3
Two year           2.8

Churn rate by internet service type:
InternetService
DSL            19.0
Fiber optic    41.9
No              7.4

Churn rate by payment method:
PaymentMethod
Bank transfer (automatic)    16.7
Credit card (automatic)      15.3
Electronic check             45.3
Mailed check                 19.2

Key finding: Month-to-month contracts show substantially higher churn than one-year
or two-year contracts, suggesting contract length is a strong retention lever.
Electronic check payment method also correlates with higher churn compared to
automatic payment methods, which may indicate lower engagement or payment friction.



## 4. Build the vector store (RAG + memory)

In [8]:
from data_setup import REPORT_PATH
from vectorstore import init_embedder, index_report, init_memory_collection

init_embedder()  # uses the proxy's gemini-embedding model

with open(REPORT_PATH) as f:
    report_text = f.read()

embedding_dim = index_report(report_text)
init_memory_collection(embedding_dim)

Gemini key loaded: True
Stored 2 chunks in Qdrant


## 5. Build the graph

In [13]:
from graph import build_graph

app = build_graph(llm_flash, llm_lite, db, df)

Graph compiled


## 6. Run it

In [10]:
from state import new_state

test_state = new_state("How many customers have churned, and what does our internal churn analysis report say about the reasons?")
final_state = app.invoke(test_state, config={"recursion_limit": 25})

print("Full path taken:", final_state["steps"])
print("\nFinal answer:\n", final_state["answer"])

Full path taken: ['supervisor→retriever(forced)', 'retriever', 'supervisor→data', 'data(sql)', 'supervisor→finish', 'generate', 'critic(approved)']

Final answer:
 Based on the provided evidence, 1,869 customers have churned. The internal churn analysis report states that the primary reasons include month-to-month contracts, a lack of tech support, and high monthly charges for customers with fiber optic internet service. 

Additionally, the report identifies that month-to-month contracts have a higher churn rate (42.7%) compared to one-year (11.3%) or two-year (2.8%) contracts, suggesting contract length is a strong retention lever. Electronic check payment methods correlate with higher churn (45.3%) compared to automatic payment methods, which may indicate lower engagement or payment friction.


## 7. Multi-turn memory test

In [14]:
from vectorstore import add_to_memory

q1 = "How many customers have churned?"
r1 = app.invoke(new_state(q1), config={"recursion_limit": 25})
print("Q1 answer:", r1["answer"])

add_to_memory(q1, r1["answer"])

q2 = "What about customers with fiber optic internet specifically?"
r2 = app.invoke(new_state(q2), config={"recursion_limit": 25})
print("Q2 answer:", r2["answer"])

Q1 answer: 1869 customers have churned.
Q2 answer: Based on the provided data, there are 708 customers with "Fiber optic" internet service. Each of these customers' details, including gender, tenure, contract type, payment method, monthly charges, total charges, and churn status, are listed in the provided database records.


In [16]:
winget install --id Git.Git -e --source winget

SyntaxError: invalid syntax (366949800.py, line 1)